# 04 — Two-Tower Test Evaluation

Loads a trained checkpoint and evaluates it on the test split using the same
ranking metrics as the ALS baseline:
- **Precision@K** — fraction of top-K recommendations that are in the test ground truth
- **Recall@K** — fraction of test ground-truth items that appear in top-K
- **NDCG@K** — graded relevance using `r = 1 + 2*is_read + beta*max(0, rating-3)`

Evaluated at K ∈ {10, 50, 100}.

Run from the `two_tower/` directory.

In [2]:
import sys
from pathlib import Path

# Resolve to two_tower/ so src/ imports work
ROOT = Path("__file__").resolve().parents[1]
sys.path.insert(0, str(ROOT))

In [3]:
import math
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import fsspec
import pyarrow.parquet as pq
from omegaconf import OmegaConf
from tqdm.auto import tqdm

from src.data.loaders import load_artifacts, build_item_feature_tensors, _iter_parquet_batches
from src.models.two_tower import TwoTowerModel

## Configuration

In [ ]:
cfg = OmegaConf.load("configs/baseline.yaml")

# ── Edit these paths ──────────────────────────────────────────────────────────
CHECKPOINT_PATH = "checkpoints/two_tower_baseline_2026-02-28_04-45-09/epoch_01_val6.9926.pt"

# MLflow run that produced the checkpoint — used to fetch vocab_and_stats.pt
MLFLOW_RUN_ID = "d37acf8b358a4e8b9609598ef511264b"

# Test split on GCS (same bucket layout as train/val)
TEST_PATH = "gs://nshen7-personal-bucket/projects/rec_sys_goodreads/data/two_tower_splits/test"

# beta used to compute interaction strength r (match ALS baseline)
BETA = 0.5

# K values to evaluate
K_VALUES = [10, 50, 100]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENCODE_BATCH_SIZE = 2048   # items/users encoded per forward pass

print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"MLflow run : {MLFLOW_RUN_ID}")
print(f"Test path  : {TEST_PATH}")
print(f"Device     : {DEVICE}")
print(f"Beta       : {BETA}")
print(f"K values   : {K_VALUES}")

## 1. Load artifacts and model

In [7]:
print("Loading artifacts...")
artifacts = load_artifacts(cfg.data.artifacts_path)
print(f"  num_users={artifacts['num_users']:,}  num_items={artifacts['num_items']:,}")

print("Building item feature tensors...")
_ITEM_FEAT_COLS = [
    "target_item_id",
    "book_primary_author_id", "book_language", "book_format", "book_top_shelves",
    "book_avg_rating", "book_ratings_count", "book_num_pages",
    "book_publication_year", "book_is_ebook",
]
item_cat_feats, item_num_feats = build_item_feature_tensors(
    _iter_parquet_batches(cfg.data.train_path, columns=_ITEM_FEAT_COLS),
    artifacts,
)
print(f"  item_cat_feats: {tuple(item_cat_feats.shape)}")
print(f"  item_num_feats: {tuple(item_num_feats.shape)}")

Loading artifacts...


FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/vocab_and_stats.pt'

In [ ]:
print("Loading model checkpoint...")
model = TwoTowerModel(cfg, artifacts).to(DEVICE)
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"  Loaded epoch {ckpt['epoch']}  (val_loss={ckpt['val_loss']:.4f})")

## 2. Load test split

In [ ]:
# Columns needed for evaluation
_TEST_COLS = [
    "user_id", "target_item_id",
    "is_read", "rating",
    "history_item_ids", "history_item_weights",
]

print("Loading test split...")
test_batches = list(_iter_parquet_batches(TEST_PATH, columns=_TEST_COLS))
test_df = pd.concat(test_batches, ignore_index=True)
del test_batches

print(f"  {len(test_df):,} interactions")
print(f"  {test_df['user_id'].nunique():,} unique users")
print(f"  {test_df['target_item_id'].nunique():,} unique items")

## 3. Compute graded relevance

Using the same formula as the ALS baseline:
$$r = 1 + 2 \cdot \text{is\_read} + \beta \cdot \max(0,\; \text{rating} - 3)$$

In [ ]:
test_df["r"] = (
    1
    + 2 * test_df["is_read"]
    + BETA * np.maximum(0, test_df["rating"] - 3)
)

# Ground-truth dict: user_id -> {item_id: r}
ground_truth: dict[int, dict[int, float]] = defaultdict(dict)
for row in test_df[["user_id", "target_item_id", "r"]].itertuples(index=False):
    ground_truth[row.user_id][row.target_item_id] = row.r

print(f"Ground truth built for {len(ground_truth):,} users")

## 4. Encode all items in the catalogue

In [ ]:
num_items = artifacts["num_items"]   # max item_id (1-indexed)
# item_ids 1..num_items (0 is the PAD item)
all_item_ids = torch.arange(1, num_items + 1, dtype=torch.long)

item_vecs_list = []
with torch.no_grad():
    for start in tqdm(range(0, len(all_item_ids), ENCODE_BATCH_SIZE), desc="Encoding items"):
        ids = all_item_ids[start : start + ENCODE_BATCH_SIZE].to(DEVICE)
        cat = item_cat_feats[ids].to(DEVICE)
        num = item_num_feats[ids].to(DEVICE)
        vecs = model.encode_items(ids, cat, num)          # [B, d_out]
        item_vecs_list.append(vecs.cpu())

# Shape: [num_items, d_out]  (row i corresponds to item_id = i+1)
item_vecs = torch.cat(item_vecs_list, dim=0)
print(f"Item embedding matrix: {tuple(item_vecs.shape)}")

## 5. Encode test users and retrieve top-K recommendations

In [ ]:
# One representative row per user (history is user-level, identical across rows)
user_rows = (
    test_df[["user_id", "history_item_ids", "history_item_weights"]]
    .drop_duplicates(subset=["user_id"])
    .reset_index(drop=True)
)
print(f"Encoding {len(user_rows):,} unique test users...")

max_k = max(K_VALUES)

# top_k_per_user[user_id] = list of top-max_k recommended item_ids (1-indexed)
top_k_per_user: dict[int, list[int]] = {}

item_vecs_device = item_vecs.to(DEVICE)  # keep on GPU for fast matmul

with torch.no_grad():
    for start in tqdm(range(0, len(user_rows), ENCODE_BATCH_SIZE), desc="Encoding users"):
        batch_rows = user_rows.iloc[start : start + ENCODE_BATCH_SIZE]

        user_ids_t = torch.tensor(batch_rows["user_id"].values, dtype=torch.long).to(DEVICE)
        history_ids_t = torch.tensor(
            np.array(batch_rows["history_item_ids"].tolist(), dtype=np.int32),
            dtype=torch.long,
        ).to(DEVICE)
        history_wts_t = torch.tensor(
            np.array(batch_rows["history_item_weights"].tolist(), dtype=np.float32),
            dtype=torch.float32,
        ).to(DEVICE)

        user_vecs_batch = model.encode_users(user_ids_t, history_ids_t, history_wts_t)  # [B, d]

        # scores[b, i] = dot(user_vec[b], item_vec[i])
        scores = user_vecs_batch @ item_vecs_device.T   # [B, num_items]

        # top-max_k item indices (0-indexed into item_vecs → item_id = idx+1)
        topk_indices = torch.topk(scores, k=max_k, dim=1).indices.cpu().numpy()  # [B, max_k]

        for i, uid in enumerate(batch_rows["user_id"].values):
            top_k_per_user[int(uid)] = (topk_indices[i] + 1).tolist()  # convert to 1-indexed item_ids

print(f"Recommendations generated for {len(top_k_per_user):,} users")

## 6. Compute ranking metrics

Identical formulas to `evaluate_ranking()` in `03_fit_als_full.py`.

In [ ]:
def compute_ranking_metrics(
    top_k_per_user: dict[int, list[int]],
    ground_truth: dict[int, dict[int, float]],
    k: int,
) -> dict[str, float]:
    """
    Compute Precision@K, Recall@K, NDCG@K (graded relevance).

    Parameters
    ----------
    top_k_per_user : dict[user_id -> list of recommended item_ids (top-max_k)]
    ground_truth   : dict[user_id -> dict[item_id -> r]]
    k              : cutoff
    """
    eval_users = list(ground_truth.keys())
    n_users = len(eval_users)

    total_hits = 0
    sum_recall = 0.0
    sum_ndcg = 0.0

    for uid in eval_users:
        recs = top_k_per_user.get(uid, [])[:k]
        gt = ground_truth[uid]          # {item_id: r}

        n_relevant = len(gt)
        n_hits = sum(1 for item_id in recs if item_id in gt)

        # --- Precision@K ---
        total_hits += n_hits

        # --- Recall@K ---
        sum_recall += n_hits / max(n_relevant, 1)

        # --- NDCG@K (graded) ---
        # DCG
        dcg = sum(
            gt[item_id] / math.log2(rank + 2)  # rank is 0-indexed
            for rank, item_id in enumerate(recs)
            if item_id in gt
        )
        # IDCG: sort ground-truth by r desc, take top-k
        sorted_r = sorted(gt.values(), reverse=True)[:k]
        idcg = sum(r / math.log2(i + 2) for i, r in enumerate(sorted_r))
        sum_ndcg += dcg / idcg if idcg > 0 else 0.0

    return {
        f"precision@{k}": total_hits / (n_users * k),
        f"recall@{k}": sum_recall / n_users,
        f"ndcg@{k}": sum_ndcg / n_users,
    }

In [ ]:
test_metrics_all = {}
for k in K_VALUES:
    t0 = time.time()
    metrics = compute_ranking_metrics(top_k_per_user, ground_truth, k=k)
    elapsed = time.time() - t0
    test_metrics_all[k] = metrics
    print(f"K={k:3d}  NDCG@K={metrics[f'ndcg@{k}']:.4f}  "
          f"P@K={metrics[f'precision@{k}']:.4f}  "
          f"R@K={metrics[f'recall@{k}']:.4f}  ({elapsed:.1f}s)")

## 7. Summary

In [ ]:
print("=" * 55)
print("  TWO-TOWER MODEL — TEST EVALUATION")
print("=" * 55)
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Test users : {len(ground_truth):,}")
print(f"Beta       : {BETA}  (r = 1 + 2*is_read + {BETA}*max(0, rating-3))")
print()
print(f"{'K':>5}  {'NDCG@K':>10}  {'Precision@K':>12}  {'Recall@K':>10}")
print("-" * 45)
for k in K_VALUES:
    m = test_metrics_all[k]
    print(f"{k:>5}  {m[f'ndcg@{k}']:>10.4f}  {m[f'precision@{k}']:>12.4f}  {m[f'recall@{k}']:>10.4f}")
print("=" * 55)

## 8. (Optional) Per-user metric distribution

In [ ]:
K_PLOT = 10  # which K to inspect

ndcg_per_user = []
recall_per_user = []

for uid, gt in ground_truth.items():
    recs = top_k_per_user.get(uid, [])[:K_PLOT]
    n_hits = sum(1 for item_id in recs if item_id in gt)
    recall_per_user.append(n_hits / max(len(gt), 1))

    dcg = sum(
        gt[item_id] / math.log2(rank + 2)
        for rank, item_id in enumerate(recs)
        if item_id in gt
    )
    sorted_r = sorted(gt.values(), reverse=True)[:K_PLOT]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(sorted_r))
    ndcg_per_user.append(dcg / idcg if idcg > 0 else 0.0)

ndcg_arr = np.array(ndcg_per_user)
recall_arr = np.array(recall_per_user)

print(f"NDCG@{K_PLOT} distribution (n={len(ndcg_arr):,} users):")
for p in [0, 10, 25, 50, 75, 90, 100]:
    print(f"  p{p:3d}: {np.percentile(ndcg_arr, p):.4f}")
print()
print(f"Recall@{K_PLOT} distribution:")
for p in [0, 10, 25, 50, 75, 90, 100]:
    print(f"  p{p:3d}: {np.percentile(recall_arr, p):.4f}")